In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

I0000 00:00:1786989145.088980  142420 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786989145.089514  142420 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786989145.145257  142420 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786989147.013152  142420 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

In [2]:
# Configurar el detector usando la nueva API de tareas
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

# wget -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task


options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7
)
detector = vision.HandLandmarker.create_from_options(options)

# Inicializar la cámara web
cap = cv2.VideoCapture(0)
print("Iniciando detector de gestos moderno. Presiona 'q' para salir.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("No se pudo acceder a la cámara.")
        break

    # Voltear la imagen horizontalmente para un efecto espejo natural y convertir a RGB
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Crear un objeto Image de MediaPipe a partir del fotograma de OpenCV
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    # Procesar la imagen con la nueva API
    detection_result = detector.detect(mp_image)
    gesture = "Desconocido"

    if detection_result.hand_landmarks:
        for hand_landmarks in detection_result.hand_landmarks:
            # Extraer puntos clave relevantes (la nueva API devuelve una lista de NormalizedLandmark)
            wrist = hand_landmarks[0]         # Muñeca
            index_tip = hand_landmarks[8]     # Punta del dedo índice
            middle_tip = hand_landmarks[12]   # Punta del dedo medio
            ring_tip = hand_landmarks[16]     # Punta del dedo anular
            pinky_tip = hand_landmarks[20]    # Punta del dedo meñique
            
            index_mcp = hand_landmarks[5]     # Nudillo índice
            middle_mcp = hand_landmarks[9]    # Nudillo medio
            ring_mcp = hand_landmarks[13]    # Nudillo anular
            pinky_mcp = hand_landmarks[17]    # Nudillo meñique
            
            # 1. Detección de STOP (Palma levantada de forma frontal)
            fingers_extended = (
                index_tip.y < index_mcp.y and
                middle_tip.y < middle_mcp.y and
                ring_tip.y < ring_mcp.y and
                pinky_tip.y < pinky_mcp.y
            )
            
            if fingers_extended:
                gesture = "Stop"
            else:
                # 2. Detección de Inclinaciones
                dx = middle_mcp.x - wrist.x
                dz = middle_mcp.z - wrist.z  # Coordenada de profundidad (Z)
                
                threshold_lat = 0.12
                threshold_depth = 0.05
                
                if dx > threshold_lat:
                    gesture = "Derecha"
                elif dx < -threshold_lat:
                    gesture = "Izquierda"
                elif dz < -threshold_depth:
                    gesture = "Adelante"
                elif dz > threshold_depth:
                    gesture = "Atrás"
                else:
                    gesture = "Neutro"

            # Mostrar el texto del gesto detectado en pantalla
            cv2.putText(frame, f"Gesto: {gesture}", (30, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

    # Mostrar la ventana de video
    cv2.imshow('Detector de Gestos con MediaPipe', frame)

    # Salir al presionar la tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786989148.773264  142522 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786989148.787453  142523 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Iniciando detector de gestos moderno. Presiona 'q' para salir.


W0000 00:00:1786989149.600766  142523 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /opt/miniconda3/envs/research/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https